# Глава 9. Алгоритмическая оптимизация
В этой главе поговорим про сущетсвующие модификации трансформерных моделей, применяющиеся для ускорения обучения и инференса. Акцент именно на алгоритмические хаки, про ускорение с помощью железа и параллелизации будем говорить в следующих главах. Обзор на методы борьбы с квадратичным вниманием, поговорим про FlashAttention и затронем спекулятивный инференс.

## Введение
В 4 главе мы отмечали, что при всех плюсах у трансформерной разитектуры есть два важных недостатка

1) Самовнимание считает попарные взаимодействия всех токенов, то есть его стоимость по вычислениям и памяти растёт как $O(n^2)$ по длине последовательности $n$. Пока контекст короткий, это незаметно, но на больших контекстах (длинных документах, репозиториях кода и многошаговых агентских диалогах) квадратичная сложность становится существенным ограничением, проявляющимся на этапе обучения и на префилле

2) Во время инференса токены порождаются по одному, каждый требует отдельного прохода модели, и на каждом шаге из медленной памяти (HBM) приходится заново вычитывать веса модели и весь накопленный KV-кэш. Здесь упираемся в пропускную способность памяти: тензорные ядра простаивают, GPU большую часть времени ждёт данные. Размер KV-кэша при этом линейно растёт с длиной контекста и размером батча

## Эффективный Attention
В 2020 году было несколько попыток победить квадратичную сложность. Базовая идея у всех общая: матрица внимания n×n на практике почти везде «лишняя», потому что softmax концентрирует вес на небольшом числе ключей, а нужные связи часто либо локальны, либо низкоранговы. Различаются методы тем, как именно они сокращают эту матрицу

### Reformer

[[Kitaev et al, 2020]](https://arxiv.org/abs/2001.04451) 

Архитектура Reformer = Reversable Transformer. Авторы из Google объединяют две идеи. 

1) внимание на основе LSH (locality-sensitive hashing): Query эмбединги и Value эмбединги хэшируются так, что близкие по скалярному произведению попадают в один бакет, и каждый токен считает внимание только внутри своего бакета<br><img src="img/reformer.png" width=600>  
2) обратимые слои (reversible residuals): активации не хранятся для обратного прохода, а пересчитываются, благодаря чему память перестаёт зависеть от глубины сети.



Reformer важен как первая попытка одновременно сэкономить и на времени, и на памяти.

### Longformer

[[Beltagi et al, 2020]](https://arxiv.org/abs/2004.05150)

Longformer = Long Document Transformer. Каждый токен видит скользящее окно соседей (локальное внимание), окна можно делать разреженными-расширенными (dilated) для увеличения рецептивного поля, а несколько специальных токенов получают глобальное внимание ко всей последовательности. Идея — большая часть нужного контекста лежит рядом, а несколько глобальных токенов переносят дальнюю информацию. Сложность становится линейной по n

<img src="img/longformer.png" width=500>

С чем они себя сравнивают:
- Transformer-XL (2019)
- Adaptive Span (2019)
- Compressive (2020)
- Reformer (2020)
- Sparse (2019)
- Routing (2020)
- BP-Transformer (2019)
- Blockwise (2019)

### BigBird

[[Zaheer, 2021]](https://arxiv.org/abs/2007.14062)

Авторы BigBird к скользящему окну и глобальным токенам добавляют случайность: каждый токен дополнительно смотрит на несколько случайно выбранных. 

<img src="img/bigbird.png" width=500>

Таким образом моделируется важный принцип из теории графов - случайные рёбра превращают разреженный граф в «Small World» граф (в теории графов так называют графы с высокой связанностью - путь между любой парой вершин короткий). Поэтому дальнее смешивание информации сохраняется почти бесплатно. Авторы показывают, что такая схема сохраняет выразительность полного внимания (является универсальным аппроксиматором), оставаясь линейной

### Routing Transformer

[[Google, 2020]](https://arxiv.org/abs/2003.05997)

В рамках __Routing Transformer__ авторы делают маску разреженности динамической, зависящей от входа. Токены кластеризуются (онлайн k-means), и каждый запрос считает внимание только с ключами своего кластера

<img src="img/routing.png" width=500>

### Linformer

[[Wang et al, 2021]](https://arxiv.org/abs/2006.04768)

__Linformer__ упаковывает сигнал в маленький вектор: последовательности K и V эмбедингов длины n проецируют обучаемыми линейными отображениями в маленькую фиксированную размерность k. Из-за того что эта размерность фиксирована, self-attention становится линейным ($O(n)$ сложность проекции + $O(1)$ сложность расчета уменьшенного внимания). 

<img src="img/linformer1.png" width=500>

Считается быстро, но ограниченный размер делает метод более грубым, чем полное внимание

### Performer

[[arXiv]](https://arxiv.org/abs/2009.14794)

__Performer__ заменяет приближение матрицы на переписывание самой формулы. softmax-внимание выражается через ядро, которое аппроксимируется случайными признаками (механизм FAVOR+ — положительные ортогональные случайные признаки). После этого можно воспользоваться ассоциативностью умножения матриц и посчитать φ(K)ᵀV раньше, чем умножать на φ(Q), и тем самым вообще не материализовать матрицу n×n. Получается несмещённая оценка обычного внимания, линейная по n, не требующая предположений о длине и работающая как замена «на лету». Это идейно красивый ход: бороться не с матрицей, а с порядком вычислений.

### RoFormer (RoPE)

[[arXiv]](https://arxiv.org/abs/2104.09864)

__RoFormer__  оптимизирует позиционное кодирование. Ключевой компонент ROPE = Rotary Position Embedding кодирует позицию поворотом векторов запроса и ключа на угол, пропорциональный позиции; после этого их скалярное произведение зависит только от относительного расстояния между токенами. RoPE ничего не добавляет в KV-кэш, встраивается прямо внутрь внимания и заметно лучше обучаемых абсолютных эмбеддингов обобщается на длины, не виденные при обучении. 

<img src="img/roformer1.png" width=500>

Один из тех методов, который стал стандартом. С использованием RoPE построены позднейшие приёмы растяжения контекста (интерполяция позиций, NTK-scaling, YaRN)

## Экономия KV-кэша
Эта линия посвящена второму узкому месту — инференсу. При генерации мы кэшируем ключи и значения всех прошлых токенов. Размер кэша пропорционален числу слоёв, числу голов, размерности головы, длине контекста и батчу — и именно он определяет, сколько памяти и пропускной способности съест декодирование. Логичный ход: уменьшить то, что хранится. Эволюция здесь идёт по нарастающей радикальности.

### MQA
[[arXiv]](https://arxiv.org/abs/1911.02150)<br>
__Multi-Query Attention__ - самая простая мысль: оставить много голов у запросов, но сделать общими одну голову ключей и одну голову значений на всю сеть. Кэш сразу уменьшается в число-голов раз, декодирование заметно ускоряется. Расплата — просадка качества и нестабильность обучения, потому что мы слишком сильно урезали разнообразие представлений K и V

<img src="img/mqa.png" width=500>

### GQA
[[arXiv]](https://arxiv.org/abs/2305.13245)<br>
__Grouped-Query Attention__ компромисс между полным многоголовым вниманием (MHA) и MQA. Головы запросов делятся на G групп, и каждая группа разделяет одну голову K/V. При G = 1 это MQA, при G = числу голов — обычный MHA; промежуточные значения дают почти качество MHA при почти экономии MQA. Важный практический бонус: существующую MHA-модель можно дёшево «дообучить» (uptrain) в GQA. Именно поэтому GQA стал стандартом в LLaMA-2/3, Mistral и многих других.

<img src="img/gqa.png" width=500>

### MLA
[[arXiv]](https://arxiv.org/abs/2405.04434)
__Multi-head Latent Attention__ (DeepSeek-V2) меняет сам вопрос. Вместо «как поделить меньшее число голов K/V» спрашивается «зачем вообще хранить полноразмерные K и V». K и V совместно сжимаются низкоранговой проекцией в маленький латентный вектор, и в кэше лежит только он; при вычислении внимания пер-головые K и V восстанавливаются обратной проекцией на лету. В DeepSeek-V2 это дало сокращение KV-кэша примерно на 93% относительно MHA, причём, по их измерениям, не ценой качества, а с небольшим выигрышем. Платой стала сложность: низкоранговое сжатие плохо дружит с RoPE, поэтому введён «расщеплённый» (decoupled) RoPE — отдельная небольшая часть размерностей несёт позиционную информацию; и приём «поглощения весов» (weight absorption), сворачивающий проекции, чтобы восстановление не стоило лишних вычислений. MLA архитектурно более инвазивен, чем GQA, но и потенциально мощнее: он торгует дополнительными вычислениями (распаковка) за резкое снижение памяти и трафика

<img src="img/mla.png" width=500>

### NSA
[[arXiv]](https://arxiv.org/abs/2502.11089)
__Native Sparse Attention__ (DeepSeek, 2025) замыкает круг, возвращая разреженность из первой волны — но в современном исполнении. Для каждого запроса работают три ветви: сжатие (грубое — блоки токенов суммаризируются в компактные представления, дающие глобальный обзор), отбор (точное — выбираются самые релевантные блоки токенов, к которым применяется полное внимание) и скользящее окно (свежий локальный контекст); их выходы смешиваются обучаемым гейтом. Два принципиальных отличия от методов 2020 года. Во-первых, NSA обучаема нативно — разреженность присутствует с самого предобучения, а не навешивается на инференсе. Во-вторых, она аппаратно-согласована: шаблон спроектирован под GPU (сбалансированная арифметическая интенсивность, блочные ядра, выровненные под группировку GQA), поэтому теоретическая экономия операций превращается в реальное ускорение по времени. На последовательностях в 64k NSA заметно быстрее полного внимания на декодировании, прямом и обратном проходах, при этом не уступая ему в качестве. По сути это синтез двух линий: разреженного длинного контекста и современной инференс-эффективности.

<img src="img/nsa.png" width=500>

Сводно по этой секции:

| Метод | Что делает с K/V | Что в кэше | Эффект |
|---|---|---|---|
| MHA | у каждой головы свои K/V | все головы | базовая точность, тяжёлый кэш |
| MQA | одна общая голова K/V | 1 голова | максимальная экономия, просадка качества |
| GQA | K/V общие внутри группы | G голов | баланс качества и кэша, отраслевой стандарт |
| MLA | низкоранговое сжатие K/V | латентный вектор | сильное сжатие при сохранении качества, сложнее реализация |
| NSA | обучаемая разреженность (сжатие + отбор + окно) | разреженный набор блоков | длинный контекст и скорость, согласовано с железом |

## FlashAttention
(Dao et al, 2022) пошли по другому пути. Они задались вопросом, можно ли перекомпоновать алгоритм расчета, оставив его точным, но ускорить вычисление. Так родился алгоритм __FlashAttention__ модификация обычного самовнимания, ускоряющая расчет. Ключевое наблюдение — стандартное внимание упирается не в вычисления, а в память: оно записывает в медленную HBM огромную промежуточную матрицу n×n и читает её обратно. Значит, надо минимизировать обращения к HBM, а не число операций.

### FlashAttention-1
[[Dao et al, 2022]](https://arxiv.org/abs/2205.14135)<Br>
FlashAttention компонует вычисление внимания таким образом, что становится более оптимальным по I/O. На GPU есть быстрая SRAM память и медленная HBM, хочется больше вычислений делать на SRAM, 

Attention матрица нарезается на куски (процесс называется тайлинг): блоки Q, K, V подгружаются из HBM в быструю память SRAM и обрабатываются по частям. 
Softmax считается «онлайн» по частям (с бегущими максимумом и суммой), поэтому полная матрица внимания нигде не материализуется целиком. 
На обратном проходе используется пересчёт: вместо хранения большой матрицы её восстанавливают из компактной статистики. 

В итоге память линейна по n, число обращений к HBM резко падает, а итоговое ускорение по времени — в 2–4 раза, почти бесплатно и с сохранением точности.

<img src="img/flash1.png" width=600>

### FlashAttention-2
[[Dao et al, 2023]](https://arxiv.org/abs/2307.08691)<br>
__FlashAttention-2__, вторая версия (2023), не меняет алгоритм, но переписывает распараллеливание. Сокращается доля «не-matmul» операций (тензорные ядра GPU считают матричное умножение во много раз быстрее, чем специальный блок, отвечающий за экспоненту в softmax), вычисление параллелится вдоль длины последовательности, а работа лучше распределяется между варпами и блоками, уменьшая трафик через разделяемую память. Это даёт ещё около двукратного ускорения и доводит утилизацию до примерно 50–70% на A100. На H100, однако, версия достигала лишь ~35%, потому что не использовала особенности нового железа

### FlashAttention-3
[[Shah et al, 2024]](https://arxiv.org/abs/2407.08608)<br>
__FlashAttention-3__, третья версия (2024), делает шаг к со-дизайну с архитектурой Hopper (H100). Три приёма: использование асинхронности (warp-specialization — одни варпы через TMA асинхронно подгружают данные, другие в это время считают на тензорных ядрах WGMMA, перекрывая память и вычисления); чередование (ping-pong) блочного matmul и softmax, чтобы медленная экспонента считалась одновременно с матричным умножением; и низкая точность FP8 с блочным квантованием и «incoherent processing», которые удерживают точность (примерно в 2,6 раза меньше ошибка, чем у наивного FP8). В результате утилизация H100 поднимается с ~35% до ~75% (порядка 740 TFLOPs в FP16, в режиме BF16 в статье сообщается до ~85%), FP8 выходит на ~1,2 PFLOPs, а скорость — в 1,5–2 раза выше второй версии. Внимание при этом по-прежнему точное.

| Версия | Главная идея | Под какое железо | Эффект |
|---|---|---|---|
| FlashAttention | тайлинг + онлайн-softmax + пересчёт, не материализуем матрицу | Ampere и далее | память O(n), ускорение 2–4× |
| FlashAttention-2 | лучшее распараллеливание, меньше не-matmul операций | A100 | ещё ~2×, до ~50–70% пика |
| FlashAttention-3 | асинхронность Hopper + FP8 + ping-pong | H100 | 1,5–2× к v2, утилизация ~75% |

Концептуально FlashAttention важен и тем, что отчасти обесценил приближённые методы первой волны: если точное внимание и так стало IO-оптимальным, то жертвовать точностью ради асимптотики во многих случаях больше незачем.

## Спекулятивное декодирование
Здесь мы возвращаемся ко второму узкому месту с другой стороны. Авторегрессионное декодирование последовательно и упирается в пропускную способность памяти: на каждый токен мы перечитываем все веса модели из HBM, а тензорные ядра простаивают. Идея спекулятивного декодирования — занять эти простаивающие ядра: дёшево «угадать» сразу несколько будущих токенов, а потом проверить их все одним параллельным проходом большой модели и принять самый длинный верный префикс. При корректной процедуре приёма результат остаётся идентичным обычной генерации (lossless), просто за один тяжёлый проход рождается несколько токенов.

[[arXiv]](https://arxiv.org/abs/2211.17192)<br>
Базовая схема (Leviathan и др., ; Chen и др., 2023, [arXiv](https://arxiv.org/abs/2302.01318)): отдельная маленькая черновая (draft) модель предлагает k токенов, большая целевая модель проверяет их параллельно, а схема приёма на основе rejection sampling гарантирует, что итоговое распределение не изменится. Выигрыш есть, если черновик дёшев, а доля принятых токенов высока. Главная проблема — нужна отдельная, хорошо согласованная с целевой, draft-модель.

### Medusa
[[Cai et al, 2024]](https://arxiv.org/abs/2401.10774)<Br>
__Medusa__ (2024) избавляется от отдельной модели. Поверх последнего скрытого состояния целевой модели добавляется несколько лёгких «голов», каждая предсказывает токен на позиции +1, +2, +3 и так далее параллельно. Множество кандидатов-продолжений проверяется сразу через древовидное внимание (tree attention). Обучать нужно только головы, это дёшево. Слабое место — головы предсказывают независимо друг от друга (предсказание второй головы не обусловлено тем, что предложила первая), поэтому совместная точность черновика падает.

<img src="img/medusa1.png" width=300>

### Hydra
[[arXiv]](https://arxiv.org/abs/2402.05109)<Br>
__Hydra__ (2024) чинит именно эту независимость: черновые головы делаются последовательно зависимыми — каждая получает на вход токены, предложенные предыдущими. По сути черновик превращается из набора независимых предсказателей в нормальную последовательную модель, что заметно повышает среднюю длину принятого фрагмента. Признаки целевой модели по-прежнему переиспользуются.

### EAGLE
[[Li et al, 2024]](https://arxiv.org/abs/2401.15077)<br>
Ключевая мысль первой версии (__EAGLE-1__, 2024): вести авторегрессию не на уровне токенов, а на уровне признаков — предпоследнего скрытого состояния, которое предсказуемее и «глаже», чем дискретные токены; затем из предсказанного признака получают токен через готовую LM-голову целевой модели. Остаточная неопределённость (какой именно токен реально выбрался при сэмплировании) снимается тем, что предыдущий выбранный токен подаётся обратно на вход маленького черновика. Более точные черновики дают более высокую долю приёма, и всё это lossless.

<img src="img/eagle1_1.png" width=500>

[[Li et al, 2024]](https://arxiv.org/abs/2406.16858)<br>
__EAGLE-2__ добавляет динамические черновые деревья: вместо фиксированного дерева кандидатов его форма подстраивается по оценкам уверенности черновика — перспективные ветви разворачиваются, маловероятные отсекаются, и за один цикл проверки принимается больше токенов.

[[Li et al, 2025]](https://arxiv.org/abs/2503.01840)<br>
__EAGLE-3__ снимает ограничение на предсказание именно признака и обучается в режиме «training-time test», то есть многошаговый процесс черновика симулируется уже на обучении, устраняя рассинхрон между обучением и инференсом. Вместо одних только верхних признаков (которые переобучаются под предсказание следующего токена) используется слияние низко-, средне- и высокоуровневых признаков. Это позволяет качеству расти с объёмом обучающих данных и даёт порядка 3–6,5× ускорения относительно обычной генерации и на 20–40% выше EAGLE-2.